In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [22]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()

        pos_enc = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        args = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pos_enc[:, 0::2] = torch.sin(position * args)
        pos_enc[:, 1::2] = torch.cos(position * args)
        pos_enc = pos_enc.unsqueeze(0)

        self.register_buffer('pe', pos_enc)

    def forward(self, x):
        """
        x: (B, T, d_model)
        возвращает positional embeddings для первых T позиций: (B, T, d_model)
        """
        T = x.size(1)
        return self.pe[:, :T, :].expand_as(x)


In [3]:
def subsequent_mask(size):
    """
    Создаёт нижне-треугольную матрицу единиц
    """
    return torch.tril(torch.ones((1, size, size), dtype=torch.bool))

def make_pad_mask(seq, pad_idx):
    """
    seq: (B, T)
    pad_idx: индекс паддинга в словаре
    возвращает mask (B, 1, T) где True для non-pad positions
    """
    return (seq != pad_idx).unsqueeze(1)


In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, d_head, dropout):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.d_head = d_head
        self.dropout = nn.Dropout(dropout)

        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_out = nn.Linear(d_model, d_model, bias=False)


    def forward(self, q, k, v, mask=None):
        B = q.size(0)

        Q = self.w_q(q)
        K = self.w_k(k)
        V = self.w_v(v)

        Q = Q.view(B, -1, self.n_head, self.d_head).transpose(1, 2)
        K = K.view(B, -1, self.n_head, self.d_head).transpose(1, 2)
        V = V.view(B, -1, self.n_head, self.d_head).transpose(1, 2)

        attention_score = Q @ K.transpose(-2, -1) / self.d_head ** 0.5

        if mask is not None:
            attention_score = attention_score.masked_fill(~mask, float('-inf'))

        attention_score = F.softmax(attention_score, dim=-1)
        attention_score = self.dropout(attention_score)

        out = attention_score @ V

        out = out.transpose(1, 2).reshape(B, -1, self.d_model)

        out = self.w_out(out)
        out = self.dropout(out)

        return out


In [5]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [11]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_head, d_ff, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_head, d_head, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attention_res = self.attention(x, x, x, mask=mask)
        x = x + self.dropout(attention_res)
        x = self.norm1(x)

        ff = self.ff(x)
        x = x + self.dropout(ff)
        x = self.norm2(x)
        
        return x

In [12]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_head, d_ff, dropout):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_head, d_head, dropout)
        self.cross_attention = MultiHeadAttention(d_model, n_head, d_head, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, mask=None):
        self_attention_res = self.self_attention(x, x, x, mask=mask)
        x = x + self.dropout(self_attention_res)
        x = self.norm1(x)
        
        cross_attention_res = self.cross_attention(x, enc_output, enc_output, mask=mask)
        x = x + self.dropout(cross_attention_res)
        x = self.norm2(x)
        
        ff = self.ff(x)
        x = x + self.dropout(ff)
        x = self.norm3(x)

        return x

In [23]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layer, n_head, d_head, d_ff, max_len, dropout, pad_idx=None):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = SinusoidalPositionalEncoding(d_model, max_len)
        self.layers = nn.Sequential(
            *[EncoderLayer(d_model, n_head, d_head, d_ff, dropout) for _ in range(n_layer)]
        )
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.pad_idx = pad_idx
    
    def forward(self, seq):
        x = self.token_embed(seq)
        x = x + self.pos_embed(x)
        
        x = self.dropout(x)
        x = self.layers(x)
        x = self.norm(x)
        
        return x

In [14]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layer, n_head, d_head, d_ff, max_len, dropout):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = SinusoidalPositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, n_head, d_head, d_ff, dropout) for _ in range(n_layer)]
        )
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, target, encoder_output):
        x = self.token_embed(target)
        x = x + self.pos_embed(x)
        
        x = self.dropout(x)
        
        _, target_size2 = target.size()
        causal_mask = subsequent_mask(target_size2).unsqueeze(0)
        
        for layer in self.layers:
            x = layer(x, encoder_output, mask=causal_mask)
        x = self.norm(x)
        
        return x

In [16]:
class Transformer(nn.Module):
    def __init__(self, vocab_size_seq, vocab_size_target, d_model=512, n_layer=6, n_head=8, d_head=64, d_ff=2048, max_len=512, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(vocab_size_seq, d_model, n_layer, n_head, d_head, d_ff, max_len, dropout)
        self.decoder = Decoder(vocab_size_target, d_model, n_layer, n_head, d_head, d_ff, max_len, dropout)
        self.out = nn.Linear(d_model, vocab_size_target)

    def forward(self, seq, target):
        encoder_output = self.encoder(seq)
        decoder_output = self.decoder(target, encoder_output)
        logits = self.out(decoder_output)
        
        return logits

In [17]:
vocab_seq = vocab_target = 28
d_model = 128
n_layer = 2
n_head = 4
d_ff = 512
max_len = 64

model = Transformer(vocab_seq, vocab_target, d_model, n_layer, n_head, d_ff, max_len, 0.1).to(device)

B = 4
T_src = 10
T_tgt = 12
seq = torch.randint(1, vocab_seq, (B, T_src), device=device)
target = torch.randint(1, vocab_target, (B, T_tgt), device=device)

logits = model(seq, target[:, :-1])
labels = target[:, 1:]

loss = F.cross_entropy(logits.view(-1, vocab_target), labels.view(-1))
print(loss.item())

TypeError: zeros() received an invalid combination of arguments - got (float, int), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


In [6]:
B = 2
T_q = 5
T_k = 7
emb = 12
heads = 3
head_size = 4
dropout = 0.0

res = MultiHeadAttention(emb_size=emb, num_heads=heads, head_size=head_size, dropout=dropout).to(device)

q = torch.randn(B, T_q, emb)
k = torch.randn(B, T_k, emb)
v = torch.randn(B, T_k, emb)

out, attn = res(q=q, k=k, v=v)

out

TypeError: MultiHeadAttention.__init__() got an unexpected keyword argument 'emb_size'